# Vantage Vision - train the obstacle model on a free GPU

This notebook runs the whole pipeline end to end (about **2 hours** on Colab's free T4 GPU):

| step | what happens | time |
|---|---|---|
| 1 | download ROD street obstacles + COCO indoor objects, merge into 29 Vantage classes | ~15 min |
| 2 | fine-tune YOLO11n (the model family the Raspberry Pi AI Camera runs) | ~90 min |
| 3 | test on unseen photos + compare with the stock camera model | ~3 min |
| 4 | quantise + compile for the IMX500 chip, ready for the Pi | ~5 min |

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 0. Get the code
The repository is private, so paste a GitHub token with *read* access when asked
(github.com -> Settings -> Developer settings -> Fine-grained tokens -> this repo, Contents: read).
Alternatively upload the `vision/` folder to `/content/Vantage-Training/vision` with the file browser on the left.

In [ ]:
import os, getpass
BRANCH = "claude/vantage-core-script-srhw18"
if not os.path.exists("/content/Vantage-Training"):
    tok = getpass.getpass("GitHub token (leave empty if the repo is public): ").strip()
    auth = f"{tok}@" if tok else ""
    !git clone -q -b {BRANCH} https://{auth}github.com/vantagehalo-bot/Vantage-Training.git /content/Vantage-Training
%cd /content/Vantage-Training
!pip -q install ultralytics huggingface_hub pyarrow

## 1. Build the dataset
* **ROD** - 24k photos from a pedestrian's eye level (stairs, manholes, poles, cones, crossings ...). MIT licence.
* **COCO** - only the photos containing indoor obstacles (tables, sofas, beds, TVs, suitcases ...). CC BY 4.0.

Each extra COCO shard adds ~1,500 useful images (~480 MB download). 4 is a good balance.

In [ ]:
!python vision/build_dataset.py --coco-shards 4

In [ ]:
# Look at a few training pictures with their labels
import random, glob, cv2, matplotlib.pyplot as plt, sys
sys.path.insert(0, "vision")
import taxonomy as tx
from draw import draw_detections
imgs = random.sample(glob.glob("vision/datasets/vantage/images/train/*.jpg"), 6)
fig, axs = plt.subplots(2, 3, figsize=(18, 9))
for ax, p in zip(axs.flat, imgs):
    im = cv2.imread(p); lbl = p.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
    dets = []
    for ln in open(lbl).read().splitlines():
        c, x, y, w, h = ln.split(); x, y, w, h = map(float, (x, y, w, h))
        dets.append((tx.NAMES[int(c)], 1.0, (x - w/2, y - h/2, x + w/2, y + h/2)))
    ax.imshow(draw_detections(im, dets)[:, :, ::-1]); ax.axis("off")
plt.tight_layout()

## 2. Train
Starts from COCO-pretrained YOLO11n and fine-tunes it on our 29 classes. Watch `mAP50` climb.
If Colab disconnects, re-run the clone cell and then `!python vision/train.py --resume`.

In [ ]:
!python vision/train.py --epochs 60 --device 0

In [ ]:
from IPython.display import Image, display
display(Image("vision/runs/vantage/results.png", width=1000))
display(Image("vision/runs/vantage/confusion_matrix_normalized.png", width=800))

## 3. Evaluate on photos the model has never seen
Compares our model with the stock COCO model that ships with the AI camera.

In [ ]:
!python vision/evaluate.py --weights vision/models/vantage.pt
display(Image("vision/reports/recall_chart.png", width=800))
for p in sorted(glob.glob("vision/reports/examples/*.jpg"))[:4]:
    display(Image(p, width=1100))

## 4. Export for the Raspberry Pi AI Camera (IMX500)
Quantises to 8-bit and compiles with Sony's toolchain. Download the zip and copy it to the Pi.

In [ ]:
!python vision/export_imx.py --weights vision/models/vantage.pt
!cd vision && zip -qr /content/vantage_model_bundle.zip models reports
from google.colab import files
files.download("/content/vantage_model_bundle.zip")

On the Pi (inside `~/Vantage-Training`):
```bash
unzip vantage_model_bundle.zip -d vision/
sudo apt install imx500-all
imx500-package -i vision/models/vantage_imx_model/packerOut.zip -o vision/models/vantage_rpk
python3 vantage_core.py --model vision/models/vantage_rpk/network.rpk \
    --labels vision/models/vantage_imx_model/labels.txt --bbox-normalization --bbox-order xy
```

## 5. (Optional) try the demo right here
Creates a temporary public link you can open on your phone.

In [ ]:
!pip -q install gradio
!python vision/demo_app.py --share